# Materi 5: Word Embedding dengan Skip-gram (Word2Vec)

Materi ini adalah lanjutan dari Materi 4. Kali ini kita akan mencoba mengubah teks berita yang sudah kita kumpulkan menjadi representasi vektor numerik menggunakan algoritma **Word2Vec** dengan arsitektur **Skip-gram**.

Berbeda dengan TF-IDF yang hanya menghitung frekuensi kata, Word2Vec (Skip-gram) menggunakan jaringan saraf tiruan (*Artificial Neural Network*) sederhana dengan 1 *hidden layer* untuk mempelajari makna kata berdasarkan konteks (kata-kata di sekitarnya).

Kita akan membandingkan dua skenario preprocessing:
1. **Skenario 1 (Dengan Pembersihan Penuh):** Teks dibersihkan, huruf kecil, dan **Stopword dihapus**.
2. **Skenario 2 (Tanpa Hapus Stopword):** Teks hanya diubah menjadi huruf kecil dan dipotong per kata (tokenization). **Stopword dibiarkan**.


In [1]:
import pandas as pd
import numpy as np
import re
from gensim.models import Word2Vec
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('output/crawling_detik.csv')
print(f"Total data: {len(df)}")
df.head(3)


Total data: 200


,ID,Isi Berita,Label
0,1,Lombok - Sejumlah pekerja melakukan pengecatan...,sport
1,2,Nagoya - Sejumlah jurnalis beraktivitas di Mai...,sport
2,3,Pebalap Spanyol David Almansa jadi yang tercep...,sport


## 1. Preprocessing Skenario 1 dan 2


In [2]:
stopwords = set(["yang", "di", "dan", "ke", "dari", "untuk", "pada", "dengan", "adalah", "itu", "ini", "dalam"])

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    return text.split()

# Skenario 1: Hapus Stopword
def preprocess_scenario_1(text):
    tokens = clean_text(text)
    return [w for w in tokens if w not in stopwords and len(w) > 2]

# Skenario 2: Tanpa Hapus Stopword
def preprocess_scenario_2(text):
    tokens = clean_text(text)
    return [w for w in tokens if len(w) > 0]

df['tokens_skenario1'] = df['Isi Berita'].apply(preprocess_scenario_1)
df['tokens_skenario2'] = df['Isi Berita'].apply(preprocess_scenario_2)

vocab_skenario1 = set([word for tokens in df['tokens_skenario1'] for word in tokens])
vocab_skenario2 = set([word for tokens in df['tokens_skenario2'] for word in tokens])

print(f"Jumlah kata unik (Skenario 1): {len(vocab_skenario1)}")
print(f"Jumlah kata unik (Skenario 2): {len(vocab_skenario2)}")

pd.DataFrame({'word': list(vocab_skenario1)}).to_csv('output/vocab_skenario1.csv', index=False)
pd.DataFrame({'word': list(vocab_skenario2)}).to_csv('output/vocab_skenario2.csv', index=False)


Jumlah kata unik (Skenario 1): 7162
Jumlah kata unik (Skenario 2): 7319


## 2. Skip-gram (Word2Vec)

Kita akan melatih model Word2Vec Skip-gram (`sg=1` di gensim) dengan mengubah-ubah:
- **Window Size (2, 3, 5)**: Seberapa jauh tebakan kata tetangganya.
- **Vector Dimension (50, 100, 200)**: Panjang vektor angka per kata.

Untuk menilai "kombinasi terbaik", kita akan menghitung vektor rata-rata setiap dokumen (kalimat), lalu melatih model klasifikasi sederhana (Logistic Regression) untuk memprediksi label kategori berita. Akurasi tertinggi menunjukkan kualitas vektor terbaik.


In [3]:
def document_vector(word2vec_model, tokens):
    valid_words = [word for word in tokens if word in word2vec_model.wv.key_to_index]
    if not valid_words:
        return np.zeros(word2vec_model.vector_size)
    
    vectors = [word2vec_model.wv[word] for word in valid_words]
    return np.mean(vectors, axis=0)

def evaluate_word2vec(tokens_col, window_sizes, vector_dims):
    results = []
    
    y = df['Label'].astype('category').cat.codes
    
    for w in window_sizes:
        for d in vector_dims:
            model = Word2Vec(sentences=df[tokens_col], vector_size=d, window=w, min_count=1, sg=1, workers=4)
            
            X = np.array([document_vector(model, tokens) for tokens in df[tokens_col]])
            
            X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
            clf = LogisticRegression(max_iter=1000)
            clf.fit(X_train, y_train)
            acc = accuracy_score(y_test, clf.predict(X_test))
            
            results.append({
                'Scenario': tokens_col,
                'Window': w,
                'Dim': d,
                'Accuracy': acc
            })
            
    return pd.DataFrame(results)

windows = [2, 3, 5]
dims = [50, 100, 200]

res1 = evaluate_word2vec('tokens_skenario1', windows, dims)
res2 = evaluate_word2vec('tokens_skenario2', windows, dims)

all_results = pd.concat([res1, res2]).sort_values(by='Accuracy', ascending=False)
all_results.to_csv('output/word2vec_evaluation_results.csv', index=False)
all_results.head(10)


,Scenario,Window,Dim,Accuracy
8,tokens_skenario1,5,200,0.975
4,tokens_skenario1,3,100,0.975
7,tokens_skenario1,5,100,0.975
6,tokens_skenario1,5,50,0.975
3,tokens_skenario1,3,50,0.950
6,tokens_skenario2,5,50,0.950
8,tokens_skenario2,5,200,0.950
7,tokens_skenario2,5,100,0.950
5,tokens_skenario1,3,200,0.925
3,tokens_skenario2,3,50,0.900


## 3. Ekspor Vektor Terbaik

Setelah menemukan kombinasi terbaik, mari kita ekstraksi vektor kalimat untuk seluruh dokumen menggunakan parameter tersebut, dan simpan sebagai CSV.


In [4]:
best_config = all_results.iloc[0]
best_scenario = best_config['Scenario']
best_w = int(best_config['Window'])
best_d = int(best_config['Dim'])

print(f"Kombinasi Terbaik: {best_scenario} dengan Window={best_w} dan Dim={best_d}")

final_w2v = Word2Vec(sentences=df[best_scenario], vector_size=best_d, window=best_w, min_count=1, sg=1)

doc_vectors = [document_vector(final_w2v, tokens) for tokens in df[best_scenario]]
df_vectors = pd.DataFrame(doc_vectors, columns=[f'v_{i}' for i in range(best_d)])
df_vectors['Label'] = df['Label']

df_vectors.to_csv('output/final_document_vectors.csv', index=False)
print("Berhasil menyimpan output/final_document_vectors.csv")
df_vectors.head(3)


Kombinasi Terbaik: tokens_skenario1 dengan Window=5 dan Dim=200
Berhasil menyimpan output/final_document_vectors.csv


,v_0,v_1,v_2,v_3,v_4,v_5,v_6,v_7,v_8,v_9,...,v_191,v_192,v_193,v_194,v_195,v_196,v_197,v_198,v_199,Label
0,-0.014375,-0.044239,0.015335,0.094334,0.176208,-0.053204,0.053493,0.171771,0.074205,0.037551,...,0.031662,-0.096121,-0.164398,0.012253,0.023808,0.042735,-0.029275,0.003507,0.065144,sport
1,-0.017928,-0.047472,0.017951,0.102705,0.191380,-0.057718,0.056313,0.191423,0.081234,0.040953,...,0.037473,-0.101896,-0.178777,0.012161,0.028168,0.048407,-0.029565,0.000875,0.072248,sport
2,-0.025838,-0.046171,0.025136,0.099333,0.190561,-0.055897,0.048189,0.163397,0.085081,0.024628,...,0.034855,-0.096428,-0.162676,0.018151,0.020935,0.056003,-0.033102,-0.003556,0.070957,sport


## Klasifikasi Dengan Naive Bayes
![Word2vec Proses Klasifikasi Naive Bayes](image/Materi2/word2vec_naivebayes_process.png)

Disini saya menggunakan Orange untuk memudahkan saya menggunakan Naive Bayes untuk klasifikasinya

dan untuk metode yang saya gunakan ada 2, yaitu : Cross Validation dan Random Sampling

### Hasil Klasifikasi Naive Bayes Menggunakan metode Cross Validation : 

![Word2vec Klasifikasi Naive Bayes Cross Validation](image/Materi2/word2vec_naivebayes_crossvalidation.png)

Di atas itu adalah gambar hasil dari klasifikasi Naive Bayes menggunakan metode Cross Validation, yang dimana itu menunjukkan :

Nilai AUC nya Sendiri 0.942, untuk model naive bayes tergolong bagus

Nilai CA nya 0.855, ini juga bagus, karna menebak dengan benar 85 dari 100 data uji

Nilai F1 0.855, modelnya menjaga rasio seimbang 

Nilai Prec 0.858, cukup presisi untuk memprediksi dan jarang memberikan tebakan salah saat memprediksi suatu kelas

Nilai Recall 0.855, bisa mendeteksi sebagian data target yang diperlukan untuk ditemukan

Nilai MCC 0.713, ini juga sejalan dengan kenyataan data asli 


### Hasil Klasifikasi Naive Bayes menggunakaan Metode Random Sampling

![Word2vec Klasifikasi Naive Bayes Random Sampling](image/Materi2/word2vec_naivebayes_randomsampling.png)

Diatas itu adalah gambar hasil klasifikasi Naive Bayes Menggunakan metode Random Sampling, yang dimaina itu menunjukkan : 

Nilai AUC 0.957 tergolong sangat bagus untuk model Naive Bayes.

Nilai CA 0.875 tergolong bagus karena model menebak dengan benar 87 dari 100 data uji.

Nilai F1 0.875 menunjukkan model menjaga rasio dengan sangat seimbang.

Nilai Prec 0.876 menunjukkan model sangat presisi dan sangat jarang memberikan tebakan salah saat memprediksi suatu kelas.

Nilai Recall 0.875 menunjukkan model berhasil mendeteksi lebih banyak data target.

Nilai MCC 0.751 menunjukkan hasil sangat sejalan dengan kenyataan data asli.


## Evaluasi Confusion Matrix
![Word2vec Convusion Matrix](image/Materi2/word2vec_confusionmatrix.png)

Diatas Adalah Gambar dari Confusion Matrix, dijelaskan di gambar atas yaitu data asli kelas finanace berjumlah 340, sedangkan model yang ku buat menebak 290 data dengan benar sebagai Finance dan salah menebak 50 data sebagai sport, begitu juga untuk kelas Sport, data asli berjumlah 340 sedangkan model yang dibuat menebak 305 data dengan benar sebagai Sport, salah 35 data sport sebagai finance
